# 3D ガウシアン電荷ポテンシャルの低ランク近似

1D Tucker ノートブックの 3D 拡張。3D Coulomb カーネル $K(r) = 1/\sqrt{r^2+\varepsilon^2}$ について

1. $N^3 \times N^3$ 行列としての SVD 近似
2. 近距離/遠距離分離 (hard mask, smoothstep) + 遠距離 SVD
3. RPCA による $K = L + S$ 分解
4. $K$ を 6-way テンソルと見た Tucker 分解（3D 固有の多重線形構造）
5. 直接畳み込み vs CG poisson solver vs 解析解

を比較する。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils.extmath import randomized_svd

from src.utils.grid import build_xyz
from src.utils.metrics import l2_error
from src.decomposition.tucker import perform_tucker, reconstruct
from src.decomposition.rpca import randomized_rpca
from src.potential.poisson_solver import poisson_solve
from src.potential.charge_potential import v_analytic_gaussian

## グリッドと 3D Coulomb カーネル

$K_{ab} = 1 / \sqrt{|x_a - x_b|^2 + \varepsilon^2}$。 $N=13$ で $K$ は $2197 \times 2197$。

In [2]:
N = 13
L = 6.0
eps = 0.05
alpha = 1.0

dx = L / N
xyz = build_xyz(N, L)
R = np.sqrt(xyz[0]**2 + xyz[1]**2 + xyz[2]**2)

grid_pts = np.stack([xyz[0].ravel(), xyz[1].ravel(), xyz[2].ravel()], axis=1)

diff = grid_pts[:, None, :] - grid_pts[None, :, :]
dists = np.sqrt((diff**2).sum(-1))
K = 1.0 / np.sqrt((diff**2).sum(-1) + eps**2)

rho_grid = np.exp(-alpha * R**2)
rho_flat = rho_grid.ravel()

V_flat = K @ rho_flat * dx**3
V_grid = V_flat.reshape(N, N, N)

print(f"K.shape = {K.shape}, mem = {K.nbytes/1e6:.1f} MB")
print(f"V max/min = {V_grid.max():.4f} / {V_grid.min():.4f}")

K.shape = (2197, 2197), mem = 38.6 MB
V max/min = 7.6443 / 1.1608


## 行列 SVD によるランク $r$ 近似

In [ ]:
U, s, Vt = np.linalg.svd(K)
print("top-10 singular values:", s[:10].round(3))

ranks = [r for r in [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024] if r < N**3]
errors_svd = []
for r in ranks:
    K_r = (U[:, :r] * s[:r]) @ Vt[:r, :]
    V_r = K_r @ rho_flat * dx**3
    errors_svd.append(np.linalg.norm(V_r - V_flat) / np.linalg.norm(V_flat))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(np.arange(len(s)), s)
axes[0].set_xlabel('index')
axes[0].set_ylabel('singular value')
axes[0].set_title('singular values of K (3D)')

axes[1].loglog(ranks, errors_svd, 'o-')
axes[1].set_xlabel('rank r')
axes[1].set_ylabel('relative l2 error of V')
axes[1].set_title('rank vs error (matrix SVD)')
axes[1].grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 近距離/遠距離 hybrid: hard mask

対角近傍の特異性が SVD の収束を遅らせるので、近距離は直接、遠距離だけ低ランクで扱う。

In [ ]:
threshold = 5 * eps
near_mask = dists <= threshold
K_near = K * near_mask
K_far = K * ~near_mask

print(f"near density = {near_mask.mean():.4f}, threshold = {threshold:.3f}")

U_far, s_far, Vt_far = np.linalg.svd(K_far)

errors_hybrid = []
for r in ranks:
    K_far_r = (U_far[:, :r] * s_far[:r]) @ Vt_far[:r, :]
    V_h = (K_near + K_far_r) @ rho_flat * dx**3
    errors_hybrid.append(np.linalg.norm(V_h - V_flat) / np.linalg.norm(V_flat))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(ranks, errors_svd, 'o-', label='SVD only')
ax.loglog(ranks, errors_hybrid, 's-', label='hybrid (near + SVD on far)')
ax.set_xlabel('rank r')
ax.set_ylabel('relative l2 error')
ax.set_title('3D: rank vs error (hard near/far)')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## smoothstep 窓による分離

hard mask は不連続性が遠距離成分の高周波成分を生むため、$C^1$ 連続な smoothstep 窓で分離。

In [ ]:
def smoothstep(u, threshold):
    t = np.clip(np.abs(u) / threshold, 0, 1)
    return 1 - 3 * t**2 + 2 * t**3

w = smoothstep(dists, threshold)
K_near_s = K * w
K_far_s = K * (1 - w)

U_fs, s_fs, Vt_fs = np.linalg.svd(K_far_s)

errors_smooth = []
for r in ranks:
    K_far_r = (U_fs[:, :r] * s_fs[:r]) @ Vt_fs[:r, :]
    V_h = (K_near_s + K_far_r) @ rho_flat * dx**3
    errors_smooth.append(np.linalg.norm(V_h - V_flat) / np.linalg.norm(V_flat))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(ranks, errors_svd, 'o-', label='SVD only')
ax.loglog(ranks, errors_hybrid, 's-', label='hard near/far')
ax.loglog(ranks, errors_smooth, '^-', label='smoothstep')
ax.set_xlabel('rank r')
ax.set_ylabel('relative l2 error')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print('top-10 far singular values (smooth):', s_fs[:10].round(4))

## Randomized RPCA による $K = L + S$ 分解

1D 同様、近傍の集中部分を $S$ にスパースに押し出して、$L$ を綺麗な低ランク化候補にする。

In [ ]:
print('randomized RPCA running...')
L_rpca, S_rpca = randomized_rpca(K, rank=80, max_iter=200, tol=1e-6, verbose=True)

rank_L = (np.linalg.svd(L_rpca, compute_uv=False) > 1e-6).sum()
print(f"L 実効ランク : {rank_L}")
print(f"S 非ゼロ率   : {(np.abs(S_rpca) > 1e-6).mean():.4f}")

U_L, s_L, Vt_L = np.linalg.svd(L_rpca)

errors_rpca = []
for r in ranks:
    if r > U_L.shape[1]:
        errors_rpca.append(np.nan)
        continue
    L_r = (U_L[:, :r] * s_L[:r]) @ Vt_L[:r, :]
    V_h = (L_r + S_rpca) @ rho_flat * dx**3
    errors_rpca.append(np.linalg.norm(V_h - V_flat) / np.linalg.norm(V_flat))

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(ranks, errors_svd, 'o-', label='SVD only')
ax.loglog(ranks, errors_hybrid, 's-', label='hard near/far')
ax.loglog(ranks, errors_smooth, '^-', label='smoothstep')
ax.loglog(ranks, errors_rpca, 'd-', label='randomized RPCA')
ax.set_xlabel('rank r')
ax.set_ylabel('relative l2 error')
ax.grid(True, which='both', alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## $K$ を 6-way テンソルとみた Tucker 分解

$K \in \mathbb{R}^{N^3 \times N^3}$ を $K \in \mathbb{R}^{N \times N \times N \times N \times N \times N}$ と読み替える。

- 行列 SVD は $N^3$ 次元の 1 軸として扱う → 各軸の構造は無視
- Tucker 分解は 6 軸を独立に低ランク化 → 3D Coulomb の多重線形構造を exploit
- ストレージ: コア $r^6$ + 因子 $6 N r$ → $r$ が小さければ行列 SVD よりはるかにコンパクト

In [ ]:
K_tensor = K.reshape(N, N, N, N, N, N)
print(f'K_tensor.shape = {K_tensor.shape}')

ranks_t = [1, 2, 3, 4, 5, 6, 8, 10, 12]
errors_tucker = []
storages_tucker = []

for r in ranks_t:
    G, factors = perform_tucker(K_tensor, ranks=[r] * 6)
    K_approx_tensor = reconstruct(G, factors)
    K_approx = K_approx_tensor.reshape(N**3, N**3)

    V_t = K_approx @ rho_flat * dx**3
    err = np.linalg.norm(V_t - V_flat) / np.linalg.norm(V_flat)
    storage = G.size + sum(f.size for f in factors)

    errors_tucker.append(err)
    storages_tucker.append(storage)
    print(
        f"r={r:>2d}  V error={err:.3e}  storage={storage:>10d} "
        f"(vs full {K.size}, ratio={storage / K.size:.2e})"
    )

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].semilogy(ranks_t, errors_tucker, 'o-')
axes[0].set_xlabel('Tucker rank r (per mode)')
axes[0].set_ylabel('relative l2 error of V')
axes[0].set_title('3D: Tucker (6-way K) rank vs error')
axes[0].grid(True, which='both', alpha=0.3)

matrix_storages = [2 * N**3 * r for r in ranks]
axes[1].loglog(storages_tucker, errors_tucker, 'o-', label='Tucker (6-way K)')
axes[1].loglog(matrix_storages, errors_svd, 's-', label='matrix SVD')
axes[1].set_xlabel('storage (params)')
axes[1].set_ylabel('relative l2 error of V')
axes[1].set_title('storage vs error')
axes[1].grid(True, which='both', alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

## カーネル断面の可視化

中心点から見た K, L, S の値を距離プロファイルとして表示。

In [ ]:
mid_idx = N**3 // 2
center_xyz = grid_pts[mid_idx]
r_from_center = np.linalg.norm(grid_pts - center_xyz[None, :], axis=1)
order = np.argsort(r_from_center)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(r_from_center[order], K[mid_idx, order], '-', lw=1.8, label='K(r)')
ax.plot(r_from_center[order], L_rpca[mid_idx, order], '--', lw=1.5, label='L (RPCA)')
ax.plot(r_from_center[order], S_rpca[mid_idx, order], ':', lw=2.0, label='S (RPCA)')
ax.set_xlabel('|r - r_center|')
ax.set_ylabel('value')
ax.set_title('kernel profile from center grid point')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

## 直接畳み込み vs CG poisson solver vs 解析解

In [ ]:
V_cg, residuals = poisson_solve(rho_grid, dx, tol=1e-10)
V_analytic = v_analytic_gaussian(R, alpha)

print(f"CG iterations            : {len(residuals)}")
print(f"||V_direct - V_an|| / ||V_an|| (interior) = {l2_error(V_grid, V_analytic, dx):.4e}")
print(f"||V_cg     - V_an|| / ||V_an|| (interior) = {l2_error(V_cg,   V_analytic, dx):.4e}")

m = N // 2
xs = xyz[0, :, m, m]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xs, V_grid[:, m, m],   '-',  lw=2, label='direct (K @ rho)')
ax.plot(xs, V_cg[:, m, m],     '--', lw=2, label='CG poisson')
ax.plot(xs, V_analytic[:, m, m], ':', lw=2, label='analytic')
ax.set_xlabel('x (y=z=0)')
ax.set_ylabel('V')
ax.set_title('centerline profile of V')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()